# Single-cell perturbations: split and baseline workflow

This notebook reuses a fixed pseudo-target split and evaluates a first PCA + Ridge baseline.

**Split:** hold out all but 17 `T cells CD4+` cell–drug pairs; keep other cell types in training; seed 42. All baselines reuse the saved pair IDs.

The JSON stores pair identities because it records **which rows are training versus validation**. Gene-level DE values remain in `de_train.parquet`; the notebook selects them into `y_train` and `y_val`.

This baseline uses cell/drug identities as input, compresses training gene-response targets with PCA, predicts PC scores with Ridge, then reconstructs gene-level predictions. It is a simple baseline, not a molecular-structure model.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "split_data.py").is_file():
    PROJECT_DIR = PROJECT_DIR / "Singlecell_perturbations"
if not (PROJECT_DIR / "split_data.py").is_file():
    raise FileNotFoundError("Open this notebook from Singlecell_perturbations or the repo root.")
sys.path.insert(0, str(PROJECT_DIR))
sys.dont_write_bytecode = True

import pandas as pd
from IPython.display import display
from split_data import load_or_create_split, apply_split, get_gene_columns
from baseline_pca_ridge import fit_predict, mrrmse

DATA_DIR = PROJECT_DIR / "open-problems-single-cell-perturbations"
TARGET_CELL_TYPE = "T cells CD4+"
N_OBSERVED = 17
RANDOM_SEED = 42
SPLIT_PATH = PROJECT_DIR / "splits" / "cd4_observed17_seed42.json"

## 1. Load labeled DE data

Each row contains one cell-type–drug pair and its gene-level DE target values.

In [ ]:
de_train = pd.read_parquet(DATA_DIR / "de_train.parquet")
id_map = pd.read_csv(DATA_DIR / "id_map.csv")
gene_cols = get_gene_columns(de_train)

print(f"Labeled pairs: {len(de_train)}")
print(f"DE target genes: {len(gene_cols)}")
print(f"Official test pairs: {len(id_map)}")
display(de_train.groupby("cell_type").size().rename("available_pairs").to_frame())

## 2. Load the fixed split and prepare inputs

The manifest contains pair IDs, not expression targets. Targets are selected from matching rows in `de_train`.

In [ ]:
split = load_or_create_split(
    de_train, SPLIT_PATH,
    target_cell_type=TARGET_CELL_TYPE,
    n_observed=N_OBSERVED,
    seed=RANDOM_SEED,
)
train_df, val_df = apply_split(de_train, split)

summary = pd.concat([
    train_df.groupby("cell_type").size().rename("train"),
    val_df.groupby("cell_type").size().rename("validation"),
], axis=1).fillna(0).astype(int)
summary["total"] = summary.sum(axis=1)
display(summary)
print(f"Train rows: {len(train_df)} | Validation rows: {len(val_df)}")

In [ ]:
feature_cols = ["cell_type", "sm_name"]
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
y_train = train_df[gene_cols].copy()
y_val = val_df[gene_cols].copy()
X_test = id_map[feature_cols].copy()
test_ids = id_map["id"].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert y_train.columns.equals(y_val.columns)

## 3. PCA + Ridge baseline

`n_components` is the number of PCs summarizing gene responses. `alpha` controls Ridge regularization; larger values shrink coefficients more. These are starting values, not tuned results.

PCA is fitted only on `y_train`. Ridge learns from training identifiers and training PC scores. Validation targets are used only for scoring.

In [ ]:
N_COMPONENTS = 50
RIDGE_ALPHA = 10.0

val_predictions = fit_predict(
    X_train=X_train,
    y_train=y_train,
    X_predict=X_val,
    n_components=N_COMPONENTS,
    alpha=RIDGE_ALPHA,
)

assert val_predictions.shape == y_val.shape
assert val_predictions.columns.equals(y_val.columns)
validation_mrrmse = mrrmse(y_val, val_predictions)
print(f"Validation MRRMSE: {validation_mrrmse:.6f}")
display(val_predictions.iloc[:5, :8])

## 4. Official test predictions after model selection

After choosing PCA dimension and Ridge alpha using validation, refit on all labeled pairs and predict the official test pairs in `id_map` order.

In [ ]:
final_X_train = de_train[feature_cols].copy()
final_y_train = de_train[gene_cols].copy()
test_predictions = fit_predict(
    X_train=final_X_train,
    y_train=final_y_train,
    X_predict=X_test,
    n_components=N_COMPONENTS,
    alpha=RIDGE_ALPHA,
)
assert len(test_predictions) == len(id_map)
assert test_predictions.columns.equals(y_train.columns)
print("Test prediction matrix:", test_predictions.shape)
print("Prediction order follows id_map.")